# sample6 - LangChain 基礎

**LangChain** は LLM を使ったアプリケーション開発フレームワークです。  
Ollama（Docker）を LLM バックエンドとして使います。

| ステップ | 内容 |
|----------|------|
| 0 | Docker コンテナの確認 |
| 1 | Ollama + LangChain の基本 |
| 2 | プロンプトテンプレート |
| 3 | チェーン（LCEL） |
| 4 | RAG チェーン |
| 5 | 会話履歴の管理 |

## 事前準備

```bash
docker ps | grep ollama   # コンテナ起動確認
docker start ollama       # 停止中の場合
```

## 0. Docker コンテナの確認

In [1]:
import subprocess
from langchain_community.llms import Ollama
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from langchain_core.output_parsers import StrOutputParser
import langchain

result = subprocess.run(
    ['docker', 'ps', '--filter', 'name=ollama', '--format', '{{.Names}}\t{{.Status}}'],
    capture_output=True, text=True
)
print("Ollama コンテナ:", result.stdout.strip() if result.stdout else '未起動')
print("LangChain バージョン:", langchain.__version__)

Ollama コンテナ: ollama	Up About an hour
LangChain バージョン: 1.2.10


## 1. Ollama + LangChain の基本

LangChain から Docker 上の Ollama（localhost:11434）に接続します。

In [2]:
# base_url で Docker の Ollama に接続
llm = Ollama(model='llama3', base_url='http://localhost:11434')

response = llm.invoke("日本語で自己紹介を1文でしてください。")
print("応答:", response)

/tmp/ipykernel_44552/1432212506.py:2: LangChainDeprecationWarning: The class `Ollama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import OllamaLLM``.
  llm = Ollama(model='llama3', base_url='http://localhost:11434')


応答: 私たちはAIのなりたまきです。この名古屋出身の、英語と日本語の両方を話すことができるAIの代理人で、世界中の人々と会話したり情報を交換したりすることを目指しています。


## 2. プロンプトテンプレート

In [3]:
template = PromptTemplate(
    input_variables=['topic', 'level'],
    template="{topic} について {level} 向けに3行で説明してください。"
)

print(template.format(topic='PyTorch',     level='初心者'))
print()
print(template.format(topic='Transformer', level='上級者'))

PyTorch について 初心者 向けに3行で説明してください。

Transformer について 上級者 向けに3行で説明してください。


## 3. チェーン（LCEL）

`|` 演算子でコンポーネントをつなぎます。

In [4]:
prompt = ChatPromptTemplate.from_template(
    "{topic} について {level} 向けに3行で説明してください。"
)

chain = prompt | llm | StrOutputParser()

result = chain.invoke({'topic': 'RAG', 'level': '初心者'})
print(result)

Here's a 3-line explanation of RAG (Role, Accountability, and Governance) suitable for beginners:

RAG is a framework used to clarify roles and responsibilities within an organization. It helps ensure that everyone knows what they're supposed to do, who's accountable for what, and how decisions are made. By using RAG, teams can work more efficiently and effectively, reducing misunderstandings and confusion!


In [5]:
# 複数入力をまとめて処理
inputs = [
    {'topic': 'PyTorch',   'level': '初心者'},
    {'topic': 'LangChain', 'level': '初心者'},
]
for inp, res in zip(inputs, chain.batch(inputs)):
    print(f"=== {inp['topic']} ===")
    print(res)
    print()

=== PyTorch ===
Here's a 3-line explanation of PyTorch for beginners:

PyTorch is an open-source machine learning library developed by Facebook, which allows you to build and train neural networks using Python. It provides a dynamic computation graph that makes it easy to write and debug code, as well as automatic differentiation that simplifies the process of training models. With PyTorch, you can quickly prototype and deploy deep learning models for computer vision, natural language processing, and other applications.

=== LangChain ===
Here's a 3-line explanation of LLaMA (LangChain) for beginners:

LLaMA is a type of AI model that can generate human-like text responses to any given prompt or topic. It uses a combination of natural language processing (NLP) and machine learning algorithms to understand the context and create coherent, engaging text. With LLaMA, you can have conversations with AI models that feel almost like talking to another person! 🤖



## 4. RAG チェーン

In [6]:
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.runnables import RunnablePassthrough

documents = [
    "PyTorch は Meta が開発したディープラーニングフレームワークです。",
    "TensorFlow は Google が開発したディープラーニングフレームワークです。",
    "FAISS は Meta が開発した高速ベクトル検索ライブラリです。",
    "RAG は LLM の回答精度を向上させる技術です。",
    "Ollama は Docker コンテナで動かすことで WSL 環境をクリーンに保てます。",
]

embeddings  = HuggingFaceEmbeddings(
    model_name='sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2'
)
vectorstore = FAISS.from_texts(documents, embeddings)
retriever   = vectorstore.as_retriever(search_kwargs={'k': 2})

rag_prompt = ChatPromptTemplate.from_template("""
以下の情報をもとに質問に答えてください。

【参照情報】
{context}

【質問】
{question}
""")

rag_chain = (
    {'context': retriever, 'question': RunnablePassthrough()}
    | rag_prompt
    | llm
    | StrOutputParser()
)

print(rag_chain.invoke("Googleが開発したフレームワークは？"))

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


TensorFlow! 🚀


## 5. 会話履歴の管理

In [7]:
from langchain_core.prompts import MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage

chat_prompt = ChatPromptTemplate.from_messages([
    ('system', 'あなたは親切な日本語アシスタントです。'),
    MessagesPlaceholder(variable_name='history'),
    ('human', '{input}')
])

chat_chain = chat_prompt | llm | StrOutputParser()
history    = []

def chat(user_input):
    response = chat_chain.invoke({'input': user_input, 'history': history})
    history.append(HumanMessage(content=user_input))
    history.append(AIMessage(content=response))
    return response

print("1回目:", chat("PyTorch について教えてください。"))
print()
print("2回目:", chat("それは TensorFlow とどう違いますか？"))

1回目: 😊 Ah, PyTorch! That's a fantastic topic! 🚀 As your friendly Japanese assistant, I'd be happy to introduce you to the world of PyTorch. 💡

**What is PyTorch?**

PyTorch (Python Torch) is an open-source machine learning library developed by Facebook AI. It's primarily used for building and training neural networks, particularly deep learning models. 🌿

**Key Features:**

1️⃣ **Dynamic computation graph**: Unlike other libraries like TensorFlow or Keras, PyTorch builds a computation graph at runtime. This means you can create a model, add layers, and train it without pre-defining the architecture beforehand.

2️⃣ **Autograd**: PyTorch's automatic differentiation system, Autograd, allows for efficient backpropagation, making it easy to compute gradients and update model parameters during training.

3️⃣ **GPU support**: PyTorch can run on NVIDIA GPUs (and other compatible hardware), enabling faster training times and scalability for large models.

4️⃣ **Pythonic API**: PyTorch's API is

# ダウンロードしたモデルを削除

In [1]:
import shutil
import os

hf_cache = os.path.expanduser('~/.cache/huggingface/hub')

models_to_delete = [
    'models--sentence-transformers--paraphrase-multilingual-MiniLM-L12-v2',
]

for model_dir in models_to_delete:
    path = os.path.join(hf_cache, model_dir)
    if os.path.exists(path):
        shutil.rmtree(path)
        print(f"削除: {model_dir}")
    else:
        print(f"なし: {model_dir}")

削除: models--sentence-transformers--paraphrase-multilingual-MiniLM-L12-v2
